# SENTRY - Training and Evaluation on Kaggle (2x T4)

Full paper workflow over **existing datasets** (tiered evaluation protocol;
the PSID-8 video benchmark is future work).

**Tiers configured in this notebook:**
- **Tier S (spatial, video):** Le2i fall dataset (hypothesis test) + fire/smoke
  video dataset (control class).
- **Tier E (event-level):** UCF-Crime test split re-taxonomized to the 8-class
  taxonomy, evaluated zero-shot with YOLO-World under two prompt conditions
  (class name vs. compositional prompts derived from the annotation schema pilot).

**Integrity protocol:** run the freeze cell BEFORE any training; the test split
is only touched once per finalized model (see `protocol/PREREGISTRATION.md`).

In [ ]:
# %% [0] Environment - pin and record versions (paper Table IV-B)
!pip -q install "ultralytics==8.3.49" wandb 2>/dev/null
import torch, ultralytics, sys, random, numpy as np, json, os, time, hashlib
print("python ", sys.version.split()[0])
print("torch  ", torch.__version__, "| cuda:", torch.version.cuda, "| gpus:", torch.cuda.device_count())
print("ultralytics", ultralytics.__version__)

SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

VERSIONS = {"python": sys.version.split()[0], "torch": torch.__version__,
            "cuda": torch.version.cuda, "ultralytics": ultralytics.__version__,
            "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
            "seed": SEED}
json.dump(VERSIONS, open("versions.json", "w"), indent=2); VERSIONS

## [1] Import the sentry-psid8 modules
Upload the repository zip as a Kaggle Dataset named `sentry-psid8` (Kaggle
auto-extracts it). The cell below adds it to `sys.path` so
`from sentry.tubes import ...` works. Alternative: `!git clone <url>` and insert
that path, or `pip install -e` (pyproject.toml included).

In [ ]:
# %% [1] Repo import + experiment configuration
REPO = "/kaggle/input/sentry-psid8/sentry-psid8"     # adjust if your slug differs
assert os.path.isdir(REPO), f"Repo not found at {REPO} - upload the zip as a Kaggle Dataset"
sys.path.insert(0, REPO)
from sentry.modules import TemporalFeatureMemory
from sentry.losses import temporal_consistency_loss
from sentry.tubes import TubeLinker, confirm_events
from sentry.metrics import event_auc, event_prf, bootstrap_ci, streams_per_gpu
from sentry.seeds import run_over_seeds, dataloader_seeding, set_all_seeds, DECLARED_SEEDS
from sentry.seeds import run_over_seeds, dataloader_seeding, set_all_seeds, DECLARED_SEEDS
import sentry
print(f"repo modules OK - sentry-psid8 v{sentry.__version__}")

CFG = dict(
    le2i_root   = "/kaggle/input/falldataset-imvia",       # Le2i (fall)
    fire_root   = "/kaggle/input/fire-and-smoke-dataset",  # fire/smoke videos
    ucf_root    = "/kaggle/input/ucf-crime-dataset",  # adjust to your dataset slug
    ucf_ann     = "/kaggle/input/ucf-crime-dataset/Temporal_Anomaly_Annotation_for_Testing_Videos.txt",
    ucf_frames  = "/kaggle/working/ucf_test_frames",   # written by the extraction cell
    ucf_limit_per_class = 10,   # 0 = no cap; time-boxed default for the 3-day schedule
    imgsz       = 640,
    model_base  = "yolov8m.pt",
    epochs_A    = 100,
    epochs_B    = 30,
    window      = 8,        # W - search {4,8,16}
    hidden_ch   = 128,      # C_h - search {64,128,256}
    lambda_tc   = 1.0,      # search {0.1,0.5,1.0,2.0}
    device      = "0,1",
)

## [2] Tier S data prep - Le2i -> clips layout + splits + freeze (Phase 1)
Run the converter with `--preview` FIRST and inspect the images: Le2i mirrors
differ in the per-frame bbox column order. If the box does not cover the
person, switch `--bbox-format` to `center_hw`.

In [ ]:
# %% [2] Le2i conversion + freeze
!python {REPO}/psid8/scripts/le2i_to_clips.py --root {CFG['le2i_root']} \
    --out /kaggle/working/le2i --preview 6 --bbox-format corners
# >>> inspect preview_*.jpg, then rerun WITHOUT --preview: <<<
# !python {REPO}/psid8/scripts/le2i_to_clips.py --root {CFG['le2i_root']} --out /kaggle/working/le2i --bbox-format corners
# !python {REPO}/psid8/scripts/build_splits.py /kaggle/working/le2i/manifest.json --out /kaggle/working/le2i/splits.json --seed 0
# !cd /kaggle/working/le2i && python {REPO}/psid8/scripts/integrity_check.py manifest.json splits.json

In [ ]:
# %% [2b] Build the Ultralytics data.yaml for Tier S (fall) from the splits
import yaml, glob, shutil
def make_yolo_layout(clips_root, splits_json, out_root):
    sp = json.load(open(splits_json))["splits"]
    for split, ids in sp.items():
        for sub in ["images", "labels"]:
            os.makedirs(f"{out_root}/{sub}/{split}", exist_ok=True)
        for cid in ids:
            for fp in glob.glob(f"{clips_root}/clips/{cid}/frames/*.jpg"):
                stem = f"{cid}__{os.path.basename(fp)}"
                os.symlink(fp, f"{out_root}/images/{split}/{stem}")
                lp = fp.replace("/frames/", "/labels/").replace(".jpg", ".txt")
                dst = f"{out_root}/labels/{split}/{stem.replace('.jpg', '.txt')}"
                if os.path.exists(lp): os.symlink(lp, dst)
                else: open(dst, "w").close()          # background frame
    # Full 8-class taxonomy head: Ultralytics requires class ids < nc, and the
    # Le2i labels use the canonical id 6 (fall). Declaring all 8 names keeps the
    # unified head consistent across corpora (fire uses id 3 in the same yaml).
    names = {0: "accident", 1: "suspicious_behavior", 2: "crime", 3: "fire",
             4: "intrusion", 5: "suspicious_object", 6: "fall", 7: "vandalism"}
    data = {"path": out_root, "train": "images/train", "val": "images/val",
            "test": "images/test", "names": names}
    yaml.safe_dump(data, open(f"{out_root}/data.yaml", "w"))
    return f"{out_root}/data.yaml"
# DATA_FALL = make_yolo_layout("/kaggle/working/le2i", "/kaggle/working/le2i/splits.json", "/kaggle/working/fall_yolo")
print("uncomment after conversion; repeat the same pattern for the fire dataset")

## [3] Stage A - frame-level training (also the YOLOv8 baseline of Table II)

In [ ]:
# %% [3] Stage A - multi-seed (pre-registered seeds [0,1,2]; resumable)
from ultralytics import YOLO

def stage_a_pipeline(seed):
    base = YOLO(CFG["model_base"])
    base.train(data=DATA_FALL, imgsz=CFG["imgsz"], epochs=CFG["epochs_A"],
               seed=seed, device=CFG["device"], project="runs",
               name=f"stageA_fall_s{seed}", deterministic=True, exist_ok=True)
    m = YOLO(f"runs/stageA_fall_s{seed}/weights/best.pt").val(data=DATA_FALL, split="val")
    return {"map50": float(m.box.map50), "map5095": float(m.box.map)}

# results, agg = run_over_seeds(stage_a_pipeline, out_dir="runs/seeds_stageA_fall")
# print("Table II cell (frame-level, fall):",
#       f"{agg['map50']['mean']:.3f} +/- {agg['map50']['std']:.3f}",
#       "CI95", agg['map50']['ci95'])
# BEST_A = "runs/stageA_fall_s0/weights/best.pt"   # seed-0 weights feed Stage B dev
# Repeat the same pattern with DATA_FIRE for the fire corpus.

## [4] SENTRY - attach the TFM and measure overhead (Sec. III-C)

In [ ]:
# %% [4] Build + sanity + overhead
from sentry.ultralytics_adapter import SentryYOLO
# yolo_a = YOLO(BEST_A)
# sentry = SentryYOLO(yolo_a, hidden_ch=CFG["hidden_ch"]).eval().cuda()
# sentry.reset_stream()
# x = torch.zeros(1, 3, CFG["imgsz"], CFG["imgsz"]).cuda()
# with torch.no_grad(): _ = sentry(x); _ = sentry(x)
# n_tfm = sum(p.numel() for p in sentry.tfm.parameters())
# n_base = sum(p.numel() for p in sentry.base.parameters())
# print(f"TFM: {n_tfm/1e6:.2f}M params (+{100*n_tfm/n_base:.1f}%)")
# print("current-frame evidence:", sentry.last_evidence)

## [5] Stage B - temporal fine-tuning (TFM + L_tc), backbone frozen
Differentiable decoding for L_tc: NMS picks indices without gradient; boxes and
scores are gathered from the raw tensor **with** gradient.

In [ ]:
# %% [5] Stage B
from ultralytics.utils import ops

def decode_with_grad(raw, conf=0.15, iou=0.7, max_det=100):
    y = raw[0] if isinstance(raw, (list, tuple)) else raw   # (1, 4+nc, N)
    with torch.no_grad():
        dets = ops.non_max_suppression(y.detach(), conf, iou, max_det=max_det)[0]
    if dets.shape[0] == 0:
        return None
    yt = y[0].T                                   # (N, 4+nc) with gradient
    xywh = torch.cat([(dets[:, :2] + dets[:, 2:4]) / 2, dets[:, 2:4] - dets[:, :2]], 1)
    d = torch.cdist(xywh[:, :2], yt[:, :2])
    idx = d.argmin(1)
    boxes = ops.xywh2xyxy(yt[idx, :4])
    scores = yt[idx, 4:]
    return boxes, scores

from torch.utils.data import DataLoader
from sentry.data import VideoClipDataset, collate_clips
# ids = json.load(open("/kaggle/working/le2i/splits.json"))["splits"]["train"]
# g, winit = dataloader_seeding(seed)   # seed comes from run_over_seeds
# ds = VideoClipDataset("/kaggle/working/le2i/clips", ids, window=CFG["window"])
# dl = DataLoader(ds, batch_size=2, shuffle=True, collate_fn=collate_clips, generator=g, worker_init_fn=winit)
# sentry.train(); sentry.freeze_base()
# opt = torch.optim.AdamW([p for p in sentry.parameters() if p.requires_grad], lr=1e-3)
# for ep in range(CFG["epochs_B"]):
#     tot, nb = 0.0, 0
#     for batch in dl:
#         opt.zero_grad(); loss = 0.0
#         for clip in batch:
#             sentry.reset_stream(); prev = None
#             for f in range(clip["images"].shape[0]):
#                 raw = sentry(clip["images"][f:f+1].cuda())
#                 dec = decode_with_grad(raw)
#                 if dec and prev:
#                     loss = loss + CFG["lambda_tc"] * temporal_consistency_loss(
#                         dec[0], dec[1], prev[0].detach(), prev[1].detach())
#                 prev = dec
#         if torch.is_tensor(loss): loss.backward(); opt.step(); tot += float(loss); nb += 1
#     print(f"epoch {ep}: mean L_tc = {tot/max(nb,1):.4f}")
# torch.save(sentry.tfm.state_dict(), "tfm_stageB.pt")

## [6] Per-class thresholds (VALIDATION only) -> alerts -> event metrics (Tier S)

In [ ]:
# %% [6] tau_c on validation + streaming inference + event evaluation
import cv2
def nms_to_dets(raw, conf_floor=0.05):
    y = raw[0] if isinstance(raw, (list, tuple)) else raw
    with torch.no_grad():
        d = ops.non_max_suppression(y, conf_floor, 0.7, max_det=100)[0].cpu().numpy()
    return [{"bbox": r[:4].tolist(), "score": float(r[4]), "class_id": int(r[5])} for r in d]

def clip_to_alerts(model, frame_paths, tau, n_c, imgsz=640):
    model.eval(); model.reset_stream()
    lk = TubeLinker(iou_thr=0.5, max_gap=5)
    for t, fp in enumerate(frame_paths):
        im = cv2.resize(cv2.imread(fp), (imgsz, imgsz))
        x = torch.from_numpy(im[:, :, ::-1].copy()).permute(2,0,1)[None].float().cuda()/255
        with torch.no_grad(): raw = model(x)
        dets = [d for d in nms_to_dets(raw) if d["score"] >= tau.get(d["class_id"], 0.25)]
        for d in dets: d["evidence"] = dict(model.last_evidence)
        lk.update(t, dets)
    return confirm_events(lk.finalize(), n_c)
# Sweep tau on VALIDATION maximizing per-class F1; freeze tau*; then produce
# alerts_val.json / gt_val.json and evaluate with event_auc / event_prf.

## [6b] Tier E - UCF-Crime re-taxonomized (zero-shot open-vocabulary)
First extraction (once; resumable - re-running skips nothing automatically,
so only run it once per session and reuse `ucf_test_frames` via Save Version),
then two prompt conditions: **class-name** vs **compositional** prompts
(derived from the schema pilot; `psid8/openvocab_prompts.json`). Detections ->
tubes -> event metrics against the official test-split temporal annotation.
Class mapping: `psid8/ucfcrime_mapping.json` (categories not in our taxonomy -
Arrest, Explosion, Normal - are excluded automatically).

In [ ]:
# %% [6a] Extract frames for the mapped UCF-Crime test videos (run once per session)
!python {REPO}/psid8/scripts/ucf_extract_test_frames.py \
    --root {CFG['ucf_root']} --annotation {CFG['ucf_ann']} \
    --mapping {REPO}/psid8/ucfcrime_mapping.json --out {CFG['ucf_frames']} \
    --fps-stride 5 --limit-per-class {CFG['ucf_limit_per_class']}
# Tip: add --dry-run first to see which/how many videos would be picked,
# without spending time on I/O; drop --limit-per-class for the full test set
# once you have confirmed folder names and timing on a small run.

In [ ]:
# %% [6b] Tier E - reads UCF-Crime .mp4 test videos directly (no frame extraction)
import glob
from ultralytics import YOLOWorld
MAP = json.load(open(f"{REPO}/psid8/ucfcrime_mapping.json"))
PROMPTS = json.load(open(f"{REPO}/psid8/openvocab_prompts.json"))
ORDER = ["accident","suspicious_behavior","crime","fire","intrusion",
         "suspicious_object","fall","vandalism"]
MODE = "compositional"          # run twice: "class" and "compositional"
FPS_STRIDE = 5                  # evaluate 1 of every 5 frames (declare in paper)

def index_videos(root):
    """Map video filename -> full path (anomaly parts + normal test videos)."""
    idx = {}
    for pat in ["Anomaly-Videos-Part-*/*/*.mp4", "Testing_Normal_Videos/*.mp4",
                "Testing_Normal_Videos/*/*.mp4"]:
        for fp in glob.glob(os.path.join(root, pat)):
            idx[os.path.basename(fp)] = fp
    return idx

def parse_temporal(path):
    """Official file: '<video>.mp4  <Class>  s1 e1 s2 e2' (frame indices, -1 = none).
    Normal test videos enter with an EMPTY event list so false alarms on normal
    footage count as FPs. Unmapped anomaly classes (Arrest, Explosion) are
    excluded entirely - declared in the paper."""
    gts = {}
    for l in open(path):
        parts = l.split()
        if len(parts) < 6:
            continue
        vid, cls = parts[0], parts[1]
        if cls == "Normal" or vid.startswith("Normal"):
            gts[vid] = []
            continue
        tax = MAP.get(cls)
        if tax is None:
            continue
        ev = []
        for s, e in [(int(parts[2]), int(parts[3])), (int(parts[4]), int(parts[5]))]:
            if s >= 0:
                ev.append({"class_id": ORDER.index(tax), "t_start": s, "t_end": e})
        gts[vid] = ev
    return gts

vocab, owner = [], []
for ci, name in enumerate(ORDER):
    ps = ([PROMPTS[name]["class_prompt"]] if MODE == "class"
          else PROMPTS[name]["compositional_prompts"])
    for p in ps: vocab.append(p); owner.append(ci)
model_ow = YOLOWorld("yolov8l-worldv2.pt"); model_ow.set_classes(vocab)

import cv2
VIDX = index_videos(CFG["ucf_root"])
gts = parse_temporal(CFG["ucf_temporal"])
print(f"annotated test videos: {len(gts)} | video files indexed: {len(VIDX)}")

alerts_all, gts_all, missing = [], [], []
for vid, gt in gts.items():
    fp = VIDX.get(vid)
    if fp is None:
        missing.append(vid); continue
    cap = cv2.VideoCapture(fp)
    lk = TubeLinker(iou_thr=0.3, max_gap=3)
    t, f_idx = 0, 0
    while True:
        ok = cap.grab()                       # cheap frame skip
        if not ok: break
        f_idx += 1
        if (f_idx - 1) % FPS_STRIDE: continue
        ok, frame = cap.retrieve()
        if not ok: break
        r = model_ow.predict(frame, conf=0.05, verbose=False)[0]
        dets = [{"bbox":[float(x) for x in b.xyxy[0]], "score":float(b.conf),
                 "class_id": owner[int(b.cls)]} for b in r.boxes]
        lk.update(t, dets); t += 1
    cap.release()
    al = confirm_events(lk.finalize(), {c: 2 for c in range(8)})
    for a in al:                              # tube time -> original frame index
        a["t_start"] *= FPS_STRIDE; a["t_end"] *= FPS_STRIDE
    alerts_all += al; gts_all += gt
if missing:
    print(f"WARNING: {len(missing)} annotated videos not found on disk, e.g. {missing[:3]}")

auc = event_auc(alerts_all, gts_all, tiou_thr=0.3)
prf = event_prf(alerts_all, gts_all, tiou_thr=0.3)
print(f"[{MODE}] Event AUC: {auc:.4f}")
for c, m in prf.items():
    if m["tp"] + m["fn"] + m["fp"]:
        print(f"  {ORDER[c]:12s} P={m['P']:.3f} R={m['R']:.3f} F1={m['F1']:.3f}")
json.dump({"mode": MODE, "fps_stride": FPS_STRIDE, "auc": auc,
           "prf": {ORDER[c]: m for c, m in prf.items()},
           "n_videos": len(gts) - len(missing), "missing": missing},
          open(f"/kaggle/working/tierE_{MODE}.json", "w"), indent=1)

## [7] Latency / FPS (Sec. V-F protocol) - 1x T4, batch=1, FP16

In [ ]:
# %% [7] Measurement
# sentry.eval().half(); sentry.reset_stream()
# x = torch.zeros(1, 3, CFG["imgsz"], CFG["imgsz"]).half().cuda()
# for _ in range(50):
#     with torch.no_grad(): sentry(x)          # warm-up
# torch.cuda.synchronize(); t0 = time.perf_counter()
# N = 500
# for _ in range(N):
#     with torch.no_grad(): sentry(x)
# torch.cuda.synchronize()
# ms = (time.perf_counter() - t0) / N * 1000
# print(f"latency: {ms:.2f} ms/frame | {1000/ms:.1f} FPS | 25-fps streams/GPU: {streams_per_gpu(1000/ms)}")

## [8] Phase 5 - TEST (once per model) and artifact export
Repeat for seeds {0,1,2}; report mean +/- std and bootstrap CIs; export
`versions.json`, integrity reports, metric JSONs and weights.
**No number enters the paper unless it lives in these files.**

In [ ]:
# %% [8] Export
os.makedirs("/kaggle/working/artifacts", exist_ok=True)
for f in ["versions.json"]:
    if os.path.exists(f): os.replace(f, f"/kaggle/working/artifacts/{f}")
print("Artifacts under /kaggle/working/artifacts - attach to the repo/Zenodo.")